In [ ]:
"""
Primary Math Exam
------------------
Asks the user for a number range and a question count, then quizzes them
on random addition/subtraction problems. Ends early if, after at least
6 questions, the failure rate exceeds 40%. Prints a final scoreboard.
"""

import random
from dataclasses import dataclass

MIN_QUESTIONS_BEFORE_EARLY_STOP = 6
FAILURE_RATE_THRESHOLD = 0.40


@dataclass
class QuestionResult:
    text: str
    correct_answer: int
    user_answer: int
    is_correct: bool


def get_answer(prompt: str) -> int | None:
    try:
        raw = input(prompt).strip()
        return int(raw)
    except ValueError:
        return None


def get_question_number() -> int:
    try:
        raw = input('\nNumber of questions: ').strip()
        n = int(raw)
        if n < 5 or n > 20:
            n = 5
            print('Number of questions must be between 5 and 20. Use default 5.')
        return n
    except ValueError:
        print('Default number of questions is 5.')
        return 5


def get_level() -> int:
    levels = {
        1: 'entry',
        2: 'basic',
        3: 'advanced',
        4: 'professional',
        5: 'expert',
        6: 'guru',
    }

    print('Select difficulty level:')
    for number, name in levels.items():
        print(f'  {number}: {name}')

    try:
        level = int(input('Enter level (1-6): '))
    except ValueError:
        level = 0
    finally:
        if level <= 0 or level > 6:
            level = random.choice([1, 2, 3, 4, 5, 6])
            print(f'Randomly selected level is {level}.')
    return level


def get_exam_settings() -> tuple[int, int]:
    """
    Ask for difficulty level and total question count.
    """
    level = get_level()
    total_questions = get_question_number()
    return level, total_questions


def generate_question(level: int) -> tuple[str, int]:
    """
    Pick two random numbers and an operation.
    """
    low = 1
    high = 20
    operators = ['+']
    if level == 2:
        low = 1
        high = 100
    elif level == 3:
        low = 1
        high = 100
        operators.append('-')
    elif level == 4:
        low = 1
        high = 100
        operators += ['-', '*']
    elif level == 5:
        low = 30
        high = 100
        operators += ['-', '*', '/']
    elif level == 6:
        low = 10
        high = 100
        operators += ['-', '*', '/']
        operator2 = random.choice(['+', '-', '*'])
    operator = random.choice(operators)
    if operator == '*':
        i = random.randint(1, 30)
        j = random.randint(1, 10)
    elif operator == '/':
        i = random.randint(1, 30)
        j = random.randint(1, 10)
        i *= j
    else:
        i = random.randint(low, high)
        j = random.randint(low, high)

    if level == 6:
        if operator2 == '*':
            if operator != '/':
                if operator == '*':
                    i = random.randint(1, 10)
                j = random.randint(1, 10)
            k = random.randint(1, 10)
        else:
            k = random.randint(low, high)

    if level >= 4:
        s = random.randint(0, 1)
        if s == 1:
            i = -i

    question_text = f'{i} {operator} {j}'

    if level == 6:
        question_text += f' {operator2} {k}'

    answer = eval(question_text)

    return question_text.replace('*', 'x'), answer


def ask_question(i_question: int, level: int) -> QuestionResult:
    question_text, answer = generate_question(level)

    user_answer = get_answer(f'Q #{i_question}: {question_text} =')

    is_correct = user_answer == answer
    if is_correct:
        print('\u2713 Correct!\n')
    else:
        print(f'\u2717 Incorrect! The answer is {answer}.\n')

    return QuestionResult(
        text=question_text,
        correct_answer=answer,
        user_answer=user_answer,
        is_correct=is_correct,
    )


def should_stop_early(correct: int, incorrect: int) -> bool:
    answered = correct + incorrect
    if answered < MIN_QUESTIONS_BEFORE_EARLY_STOP:
        return False
    failure_rate = incorrect / answered
    return failure_rate > FAILURE_RATE_THRESHOLD


def print_final_results(results: list[QuestionResult]) -> None:
    correct = sum(1 for r in results if r.is_correct)
    incorrect = len(results) - correct
    answered = len(results)

    print('\n' + '=' * 40)
    print('             TEST RESULTS')
    print('=' * 40)
    print(f'Questions answered : {answered}')
    print(f'Correct            : {correct}')
    print(f'Incorrect          : {incorrect}')
    if answered > 0:
        percentage = correct / answered * 100
        print(f'Score              : {percentage:.1f}%')

    print('\nQuestion results:')
    for number, result in enumerate(results, start=1):
        mark = '\u2713' if result.is_correct else '\u2717'
        print(
            f'Question {number}: {result.text} = {result.correct_answer}  '
            f'(you said {result.user_answer})  {mark}'
        )
    print('=' * 40)


def run_math_exam() -> None:
    print('=' * 40)
    print('        PRIMARY MATH EXAM')
    print('=' * 40)

    level, total_questions = get_exam_settings()

    results: list[QuestionResult] = []
    n_correct = 0
    n_incorrect = 0

    print("\nLet's start!")
    print('-' * 40)

    try:
        for i_question in range(1, total_questions + 1):
            result = ask_question(i_question, level)
            results.append(result)

            if result.is_correct:
                n_correct += 1
            else:
                n_incorrect += 1

            if should_stop_early(n_correct, n_incorrect):
                answered = n_correct + n_incorrect
                print('\n\u26a0 Test ended early!!!')
                print(
                    f'You answered {n_correct} correctly out of {answered} questions.'
                )
                print(f'Score is under {1 - FAILURE_RATE_THRESHOLD:.0%}.')
                break
    except (KeyboardInterrupt, EOFError):
        print('\n\nExam interrupted early.')

    print_final_results(results)


if __name__ == '__main__':
    run_math_exam()


        PRIMARY MATH EXAM
Select difficulty level:
  1: entry
  2: basic
  3: advanced
  4: professional
  5: expert
  6: guru


Enter level (1-6):  6

Number of questions:  5



Let's start!
----------------------------------------


Q #1: -95 + 44 - 86 = -137


✓ Correct!



...

✓ Correct!



Q #4: -57 - 21 - 34 = -112


✓ Correct!



Q #5: 105 / 7 - 19 = -4


✓ Correct!


             TEST RESULTS
Questions answered : 5
Correct            : 5
Incorrect          : 0
Score              : 100.0%

Question results:
Question 1: -95 + 44 - 86 = -137  (you said -137)  ✓
Question 2: -20 + 25 + 59 = 64  (you said 64)  ✓
Question 3: -30 x 10 - 22 = -322  (you said -322)  ✓
Question 4: -57 - 21 - 34 = -112  (you said -112)  ✓
Question 5: 105 / 7 - 19 = -4.0  (you said -4)  ✓
